# Loan Default Classification – 0.87+ F1 Push

Notebook structure: **Data → OOF/Threshold Framework → Baselines → Encoding Upgrade → Feature Engineering → Imbalance → Tuning → Ensemble → Plots → Final Submission**.


In [ ]:
# Stage 0 - Environment setup (dependency guards) and imports
import importlib.util
import subprocess
import sys
import time
import warnings
warnings.filterwarnings("ignore")


def ensure_package(import_name, pip_name=None):
    """Install package only when missing."""
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
    else:
        print(f"{pip_name} already available.")

# Core packages often available, but keep robust for Kaggle/Colab
for imp, pipn in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("sklearn", "scikit-learn"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("imblearn", "imbalanced-learn"),
    ("scipy", "scipy"),
]:
    ensure_package(imp, pipn)

CATBOOST_AVAILABLE = True
try:
    ensure_package("catboost", "catboost")
except Exception as e:
    CATBOOST_AVAILABLE = False
    print(f"CatBoost installation unavailable: {e}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import randint, uniform
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    f1_score,
    average_precision_score,
    precision_recall_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from imblearn.over_sampling import SMOTE

if CATBOOST_AVAILABLE:
    from catboost import CatBoostClassifier

RANDOM_STATE = 42
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
np.random.seed(RANDOM_STATE)


## Stage 0 — Load data (preserve original Drive logic) + submission format checks


In [ ]:
# Read in training data (original logic preserved)
drive_url = "https://drive.google.com/file/d/1J70Sz3_t7znOFZaDHe3SEtpJ69qCUyZy/view?usp=sharing"
file_id = drive_url.split('/d/')[1].split('/')[0]
download_url = f"https://drive.google.com/uc?id={file_id}"
df_train = pd.read_csv(download_url)

# Read in test data (original logic preserved)
drive_url = "https://drive.google.com/file/d/1X7Ezau9dfp1BKYyolYEVZzGQqobtEpGn/view?usp=sharing"
file_id = drive_url.split('/d/')[1].split('/')[0]
download_url = f"https://drive.google.com/uc?id={file_id}"
df_test = pd.read_csv(download_url)

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)
assert "loan_status" in df_train.columns, "train must include loan_status"
assert "loan_status" not in df_test.columns, "test must not include loan_status"

# Sample submission alignment logic
sample_sub = None
for candidate in ["sample_submission.csv", "SampleSubmission.csv", "sample_submit.csv"]:
    try:
        sample_sub = pd.read_csv(candidate)
        print(f"Using sample submission: {candidate}")
        break
    except Exception:
        pass

if sample_sub is None:
    # fallback: preserve competition target naming
    id_col = df_test.columns[0]
    sample_sub = pd.DataFrame({id_col: df_test[id_col].values, "loan_status": 0})
    print("sample_submission.csv not found. Using fallback format with test first column + loan_status")

target_col = "loan_status"
X = df_train.drop(columns=[target_col]).copy()
y = df_train[target_col].astype(int).copy()
X_test = df_test.copy()

num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
print(f"Numeric columns: {len(num_cols)} | Categorical columns: {len(cat_cols)}")


## Stage 1 — Unified OOF evaluation framework (F1-first, PR-AUC, thresholding)


In [ ]:
THRESHOLDS = np.round(np.arange(0.05, 0.951, 0.01), 2)


def best_threshold_from_proba(y_true, proba, thresholds=THRESHOLDS):
    f1s = [f1_score(y_true, (proba >= t).astype(int)) for t in thresholds]
    idx = int(np.argmax(f1s))
    return float(thresholds[idx]), float(f1s[idx]), pd.DataFrame({"threshold": thresholds, "f1": f1s})


def evaluate_sklearn_oof(model, X, y, cv, model_name="model"):
    n = len(X)
    oof_proba = np.zeros(n)
    fold_thresholds = []
    fold_scores = []
    t0 = time.time()

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), start=1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        m = clone(model)
        m.fit(X_tr, y_tr)
        proba_va = m.predict_proba(X_va)[:, 1]
        oof_proba[va_idx] = proba_va

        t_fold, f_fold, _ = best_threshold_from_proba(y_va, proba_va)
        fold_thresholds.append(t_fold)
        fold_scores.append(f_fold)

    elapsed = time.time() - t0
    f1_05 = f1_score(y, (oof_proba >= 0.5).astype(int))
    t_global, f1_global, curve_df = best_threshold_from_proba(y, oof_proba)
    t_foldwise = float(np.median(fold_thresholds))
    f1_foldwise = f1_score(y, (oof_proba >= t_foldwise).astype(int))
    pr_auc = average_precision_score(y, oof_proba)

    return {
        "model_name": model_name,
        "oof_proba": oof_proba,
        "f1@0.5": f1_05,
        "best_threshold_global": t_global,
        "f1@best_global": f1_global,
        "best_threshold_foldwise_median": t_foldwise,
        "f1@foldwise_median": f1_foldwise,
        "pr_auc": pr_auc,
        "fold_best_thresholds": fold_thresholds,
        "fold_best_f1": fold_scores,
        "threshold_curve": curve_df,
        "train_time_sec": elapsed,
    }


def evaluate_catboost_oof(X, y, X_test, cat_cols, params=None, cv=CV, model_name="catboost"):
    params = params or {}
    default_params = dict(
        loss_function="Logloss",
        eval_metric="F1",
        random_state=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,
    )
    default_params.update(params)

    n = len(X)
    oof_proba = np.zeros(n)
    test_fold_proba = []
    fold_thresholds = []
    fold_scores = []
    t0 = time.time()

    X_cat = X.copy()
    X_test_cat = X_test.copy()
    for c in cat_cols:
        X_cat[c] = X_cat[c].astype(str)
        X_test_cat[c] = X_test_cat[c].astype(str)

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X_cat, y), start=1):
        X_tr, X_va = X_cat.iloc[tr_idx], X_cat.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model = CatBoostClassifier(**default_params)
        model.fit(
            X_tr,
            y_tr,
            cat_features=cat_cols,
            eval_set=(X_va, y_va),
            use_best_model=True,
            early_stopping_rounds=200,
        )
        proba_va = model.predict_proba(X_va)[:, 1]
        oof_proba[va_idx] = proba_va
        test_fold_proba.append(model.predict_proba(X_test_cat)[:, 1])

        t_fold, f_fold, _ = best_threshold_from_proba(y_va, proba_va)
        fold_thresholds.append(t_fold)
        fold_scores.append(f_fold)

    elapsed = time.time() - t0
    f1_05 = f1_score(y, (oof_proba >= 0.5).astype(int))
    t_global, f1_global, curve_df = best_threshold_from_proba(y, oof_proba)
    t_foldwise = float(np.median(fold_thresholds))
    f1_foldwise = f1_score(y, (oof_proba >= t_foldwise).astype(int))
    pr_auc = average_precision_score(y, oof_proba)

    return {
        "model_name": model_name,
        "oof_proba": oof_proba,
        "test_proba": np.mean(np.column_stack(test_fold_proba), axis=1),
        "f1@0.5": f1_05,
        "best_threshold_global": t_global,
        "f1@best_global": f1_global,
        "best_threshold_foldwise_median": t_foldwise,
        "f1@foldwise_median": f1_foldwise,
        "pr_auc": pr_auc,
        "fold_best_thresholds": fold_thresholds,
        "fold_best_f1": fold_scores,
        "threshold_curve": curve_df,
        "train_time_sec": elapsed,
    }


def summarize_results(results):
    return pd.DataFrame([
        {
            "model": r["model_name"],
            "F1@0.5": r["f1@0.5"],
            "F1@global_best(A)": r["f1@best_global"],
            "F1@foldwise_median(B)": r["f1@foldwise_median"],
            "PR-AUC": r["pr_auc"],
            "best_threshold_global": r["best_threshold_global"],
            "best_threshold_foldwise_median": r["best_threshold_foldwise_median"],
            "train_time_sec": r["train_time_sec"],
        }
        for r in results
    ]).sort_values("F1@global_best(A)", ascending=False)


## Stage 2 — Leakage-safe baselines (ColumnTransformer + Pipeline)


In [ ]:
numeric_pipe_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
numeric_pipe_plain = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess_scaled = ColumnTransformer([
    ("num", numeric_pipe_scaled, num_cols),
    ("cat", categorical_pipe, cat_cols)
])
preprocess_plain = ColumnTransformer([
    ("num", numeric_pipe_plain, num_cols),
    ("cat", categorical_pipe, cat_cols)
])

baseline_models = {
    "logreg_balanced": Pipeline([
        ("prep", preprocess_scaled),
        ("clf", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "histgb": Pipeline([
        ("prep", preprocess_plain),
        ("clf", HistGradientBoostingClassifier(random_state=RANDOM_STATE))
    ]),
}

baseline_results = []
for name, model in baseline_models.items():
    baseline_results.append(evaluate_sklearn_oof(model, X, y, CV, model_name=name))

baseline_table = summarize_results(baseline_results)
print("Baseline results table:")
display(baseline_table)


## Stage 3 — Encoding upgrade ablation (OneHot vs CatBoost / Target Encoding fallback)


In [ ]:
class TargetEncoderCVSafe(BaseEstimator, TransformerMixin):
    """Simple target encoder used inside CV-safe pipeline (fit called per fold only)."""
    def __init__(self, cols=None, smoothing=20.0):
        self.cols = cols
        self.smoothing = smoothing

    def fit(self, X, y):
        X = X.copy()
        y = pd.Series(y)
        self.cols_ = self.cols if self.cols is not None else X.select_dtypes(exclude=["number"]).columns.tolist()
        self.global_mean_ = float(y.mean())
        self.maps_ = {}
        for c in self.cols_:
            stat = pd.DataFrame({"x": X[c].astype(str), "y": y}).groupby("x")["y"].agg(["mean", "count"])
            smooth = (stat["count"] * stat["mean"] + self.smoothing * self.global_mean_) / (stat["count"] + self.smoothing)
            self.maps_[c] = smooth.to_dict()
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.cols_:
            X[c] = X[c].astype(str).map(self.maps_[c]).fillna(self.global_mean_)
        return X

te_model = Pipeline([
    ("target_enc", TargetEncoderCVSafe(cols=cat_cols, smoothing=25.0)),
    ("num_imp", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(
        n_estimators=700,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ))
])

encoding_results = []
# include strongest one-hot baseline
encoding_results.append(max(baseline_results, key=lambda d: d["f1@best_global"]))

if CATBOOST_AVAILABLE:
    cat_params = dict(
        depth=8,
        learning_rate=0.05,
        iterations=3000,
        l2_leaf_reg=8,
        border_count=128,
    )
    cat_res = evaluate_catboost_oof(X, y, X_test, cat_cols, params=cat_params, cv=CV, model_name="catboost_native")
    encoding_results.append(cat_res)
else:
    te_res = evaluate_sklearn_oof(te_model, X, y, CV, model_name="target_encoding_rf")
    encoding_results.append(te_res)

encoding_table = summarize_results(encoding_results)
print("Encoding ablation table:")
display(encoding_table)

best_stage3 = max(encoding_results, key=lambda d: d["f1@best_global"])
print(f"Stage 3 best: {best_stage3['model_name']}")


## Stage 4 — High-value feature engineering (log1p + missing indicators + interactions)


In [ ]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.log_cols = ["person_income", "loan_amnt", "cb_person_cred_hist_length", "loan_int_rate"]
        self.missing_cols = []

    def fit(self, X, y=None):
        X = X.copy()
        self.missing_cols = [c for c in X.columns if X[c].isna().mean() > 0.05]
        self.home_mode_ = X["person_home_ownership"].mode().iloc[0] if "person_home_ownership" in X.columns else None
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.log_cols:
            if c in X.columns:
                X[f"{c}_log1p"] = np.log1p(np.clip(X[c], a_min=0, a_max=None))

        for c in self.missing_cols:
            X[f"{c}_is_missing"] = X[c].isna().astype(int)

        if set(["loan_amnt", "person_income"]).issubset(X.columns):
            X["loan_to_income_ratio"] = X["loan_amnt"] / (X["person_income"].fillna(0) + 1)

        if set(["credit_score", "cb_person_cred_hist_length"]).issubset(X.columns):
            X["score_to_history_ratio"] = X["credit_score"] / (X["cb_person_cred_hist_length"].fillna(0) + 1)

        if set(["person_emp_exp", "loan_int_rate"]).issubset(X.columns):
            X["emp_x_int_rate"] = X["person_emp_exp"].fillna(0) * X["loan_int_rate"].fillna(0)

        if "person_home_ownership" in X.columns and "person_emp_exp" in X.columns:
            mode_home = self.home_mode_ if self.home_mode_ is not None else "MORTGAGE"
            X["emp_x_home_mode"] = X["person_emp_exp"].fillna(0) * (X["person_home_ownership"].astype(str) == str(mode_home)).astype(int)

        return X


def build_fe_onehot_model(base_clf):
    fe_num_cols = list(X.columns)
    return Pipeline([
        ("fe", FeatureEngineer()),
        ("prep", ColumnTransformer([
            ("num", Pipeline([
                ("imp", SimpleImputer(strategy="median")),
                ("sc", StandardScaler())
            ]), make_num_cols_after_fe(X)),
            ("cat", Pipeline([
                ("imp", SimpleImputer(strategy="most_frequent")),
                ("oh", OneHotEncoder(handle_unknown="ignore"))
            ]), make_cat_cols_after_fe(X)),
        ])),
        ("clf", base_clf)
    ])

# utility for dynamic columns after FE
_tmp = FeatureEngineer().fit_transform(X.copy())
def make_num_cols_after_fe(X_):
    t = FeatureEngineer().fit_transform(X_.copy())
    return t.select_dtypes(include=["number"]).columns.tolist()

def make_cat_cols_after_fe(X_):
    t = FeatureEngineer().fit_transform(X_.copy())
    n = t.select_dtypes(include=["number"]).columns.tolist()
    return [c for c in t.columns if c not in n]

if CATBOOST_AVAILABLE and best_stage3["model_name"].startswith("catboost"):
    X_fe = FeatureEngineer().fit_transform(X.copy())
    X_test_fe = FeatureEngineer().fit(X.copy()).transform(X_test.copy())
    fe_cat_cols = [c for c in X_fe.columns if c not in X_fe.select_dtypes(include=["number"]).columns]
    fe_params = dict(depth=8, learning_rate=0.05, iterations=3500, l2_leaf_reg=8, border_count=128)
    stage4_res = evaluate_catboost_oof(X_fe, y, X_test_fe, fe_cat_cols, params=fe_params, cv=CV, model_name="catboost+FE")
else:
    fe_model = Pipeline([
        ("fe", FeatureEngineer()),
        ("prep", ColumnTransformer([
            ("num", Pipeline([
                ("imp", SimpleImputer(strategy="median")),
                ("sc", StandardScaler())
            ]), make_num_cols_after_fe(X)),
            ("cat", Pipeline([
                ("imp", SimpleImputer(strategy="most_frequent")),
                ("oh", OneHotEncoder(handle_unknown="ignore"))
            ]), make_cat_cols_after_fe(X)),
        ])),
        ("clf", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE))
    ])
    stage4_res = evaluate_sklearn_oof(fe_model, X, y, CV, model_name="onehot+FE_logreg")

fe_ablation_table = summarize_results([best_stage3, stage4_res])
print("Feature engineering ablation:")
display(fe_ablation_table)


## Stage 5 — Imbalance handling (class_weight/scale_pos_weight vs SMOTE)


In [ ]:
imbalance_results = [stage4_res]

smote_model = ImbPipeline([
    ("fe", FeatureEngineer()),
    ("prep", ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler())
        ]), make_num_cols_after_fe(X)),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), make_cat_cols_after_fe(X)),
    ])),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("clf", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
])

smote_res = evaluate_sklearn_oof(smote_model, X, y, CV, model_name="onehot+FE+SMOTE_logreg")
imbalance_results.append(smote_res)
imbalance_table = summarize_results(imbalance_results)
print("Imbalance ablation:")
display(imbalance_table)

best_stage5 = max(imbalance_results, key=lambda d: d["f1@best_global"])


## Stage 6 — Competition-style hyperparameter tuning (RandomizedSearchCV)


In [ ]:
# We tune a strong sklearn path for portability; CatBoost path tuned manually via ranges if available.

search_model = ImbPipeline([
    ("fe", FeatureEngineer()),
    ("prep", ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler())
        ]), make_num_cols_after_fe(X)),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), make_cat_cols_after_fe(X)),
    ])),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced_subsample"))
])

param_dist = {
    "clf__n_estimators": randint(300, 1400),
    "clf__max_depth": randint(4, 22),
    "clf__min_samples_split": randint(2, 20),
    "clf__min_samples_leaf": randint(1, 10),
    "clf__max_features": ["sqrt", "log2", 0.5, 0.8],
}

n_iter_search = 40
search = RandomizedSearchCV(
    estimator=search_model,
    param_distributions=param_dist,
    n_iter=n_iter_search,
    scoring="f1",
    cv=CV,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)
search.fit(X, y)

best_tuned_model = search.best_estimator_
tuned_res = evaluate_sklearn_oof(best_tuned_model, X, y, CV, model_name="tuned_rf_smote_fe")

before_after_table = summarize_results([best_stage5, tuned_res])
print("Before vs After tuning:")
display(before_after_table)


## Stage 7 — OOF probability ensembling (2~3 diverse models)


In [ ]:
ensemble_candidates = []

# candidate 1: tuned model
ensemble_candidates.append(tuned_res)

# candidate 2: best baseline (usually very different bias)
ensemble_candidates.append(max(baseline_results, key=lambda d: d["f1@best_global"]))

# candidate 3: catboost path if available
if CATBOOST_AVAILABLE:
    ensemble_candidates.append(stage4_res)

# align and average OOF probabilities
oof_stack = np.column_stack([c["oof_proba"] for c in ensemble_candidates])
ensemble_oof = oof_stack.mean(axis=1)
ens_t, ens_f1, ens_curve = best_threshold_from_proba(y, ensemble_oof)
ens_pr_auc = average_precision_score(y, ensemble_oof)
ens_f1_05 = f1_score(y, (ensemble_oof >= 0.5).astype(int))
ens_fold_t = float(np.median([c["best_threshold_foldwise_median"] for c in ensemble_candidates]))
ens_f1_fold = f1_score(y, (ensemble_oof >= ens_fold_t).astype(int))

ensemble_res = {
    "model_name": "oof_mean_ensemble",
    "oof_proba": ensemble_oof,
    "f1@0.5": ens_f1_05,
    "best_threshold_global": ens_t,
    "f1@best_global": ens_f1,
    "best_threshold_foldwise_median": ens_fold_t,
    "f1@foldwise_median": ens_f1_fold,
    "pr_auc": ens_pr_auc,
    "threshold_curve": ens_curve,
    "train_time_sec": np.nan,
}

single_best = max(ensemble_candidates, key=lambda d: d["f1@best_global"])
ensemble_compare_table = summarize_results([single_best, ensemble_res])
print("Single best vs ensemble:")
display(ensemble_compare_table)

use_ensemble = ensemble_res["f1@best_global"] >= single_best["f1@best_global"]
final_choice = ensemble_res if use_ensemble else single_best
print("Final approach:", "ensemble" if use_ensemble else "single model")


## Stage 8 — Final plots, full-train fit, and submission generation


In [ ]:
# Plot 1: F1 vs threshold
curve_df = final_choice["threshold_curve"]
plt.figure(figsize=(7,4))
plt.plot(curve_df["threshold"], curve_df["f1"], lw=2)
plt.axvline(final_choice["best_threshold_global"], color="r", ls="--", label=f"best={final_choice['best_threshold_global']:.2f}")
plt.title("Final OOF F1 vs Threshold")
plt.xlabel("Threshold")
plt.ylabel("F1")
plt.legend()
plt.show()

# Plot 2: PR curve
precision, recall, _ = precision_recall_curve(y, final_choice["oof_proba"])
plt.figure(figsize=(6,5))
plt.plot(recall, precision, lw=2)
plt.title(f"PR Curve (AP={final_choice['pr_auc']:.4f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.show()

# Plot 3: confusion matrix at chosen threshold
y_hat = (final_choice["oof_proba"] >= final_choice["best_threshold_global"]).astype(int)
cm = confusion_matrix(y, y_hat)
ConfusionMatrixDisplay(cm).plot(cmap="Blues")
plt.title("Confusion Matrix @ Best Global Threshold")
plt.show()

# Plot 4: simple learning curve (train size vs F1)
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
train_scores, val_scores = [], []

base_for_curve = baseline_models["logreg_balanced"]
for frac in fractions:
    fold_train, fold_val = [], []
    for tr_idx, va_idx in CV.split(X, y):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]

        n_sub = int(len(X_tr) * frac)
        sub_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_tr), size=n_sub, replace=False)
        X_sub, y_sub = X_tr.iloc[sub_idx], y_tr.iloc[sub_idx]

        m = clone(base_for_curve)
        m.fit(X_sub, y_sub)
        p_tr = m.predict_proba(X_sub)[:, 1]
        p_va = m.predict_proba(X_va)[:, 1]

        t_sub, _, _ = best_threshold_from_proba(y_sub, p_tr)
        fold_train.append(f1_score(y_sub, (p_tr >= t_sub).astype(int)))
        fold_val.append(f1_score(y_va, (p_va >= t_sub).astype(int)))
    train_scores.append(np.mean(fold_train))
    val_scores.append(np.mean(fold_val))

plt.figure(figsize=(7,4))
plt.plot(fractions, train_scores, marker="o", label="Train F1")
plt.plot(fractions, val_scores, marker="o", label="Validation F1")
plt.title("Learning Curve (F1)")
plt.xlabel("Training fraction")
plt.ylabel("F1")
plt.legend()
plt.show()

# ---- Full data fit and test inference ----
if use_ensemble:
    test_probas = []

    # retrain tuned model
    m1 = clone(best_tuned_model)
    m1.fit(X, y)
    test_probas.append(m1.predict_proba(X_test)[:, 1])

    # retrain baseline best
    b_best_name = max(baseline_results, key=lambda d: d["f1@best_global"])["model_name"]
    m2 = clone(baseline_models[b_best_name])
    m2.fit(X, y)
    test_probas.append(m2.predict_proba(X_test)[:, 1])

    if CATBOOST_AVAILABLE:
        X_cat = X.copy(); X_test_cat = X_test.copy()
        for c in cat_cols:
            X_cat[c] = X_cat[c].astype(str)
            X_test_cat[c] = X_test_cat[c].astype(str)
        cat_final = CatBoostClassifier(
            depth=8, learning_rate=0.05, iterations=3500, l2_leaf_reg=8,
            border_count=128, loss_function="Logloss", eval_metric="F1",
            random_state=RANDOM_STATE, verbose=False, allow_writing_files=False
        )
        cat_final.fit(X_cat, y, cat_features=cat_cols)
        test_probas.append(cat_final.predict_proba(X_test_cat)[:, 1])

    final_test_proba = np.mean(np.column_stack(test_probas), axis=1)
else:
    if final_choice["model_name"] == "tuned_rf_smote_fe":
        final_model = clone(best_tuned_model)
        final_model.fit(X, y)
        final_test_proba = final_model.predict_proba(X_test)[:, 1]
    elif final_choice["model_name"].startswith("catboost") and CATBOOST_AVAILABLE:
        X_cat = X.copy(); X_test_cat = X_test.copy()
        for c in cat_cols:
            X_cat[c] = X_cat[c].astype(str)
            X_test_cat[c] = X_test_cat[c].astype(str)
        final_model = CatBoostClassifier(
            depth=8, learning_rate=0.05, iterations=3500, l2_leaf_reg=8,
            border_count=128, loss_function="Logloss", eval_metric="F1",
            random_state=RANDOM_STATE, verbose=False, allow_writing_files=False
        )
        final_model.fit(X_cat, y, cat_features=cat_cols)
        final_test_proba = final_model.predict_proba(X_test_cat)[:, 1]
    else:
        # fallback to best baseline model
        b_best_name = max(baseline_results, key=lambda d: d["f1@best_global"])["model_name"]
        final_model = clone(baseline_models[b_best_name])
        final_model.fit(X, y)
        final_test_proba = final_model.predict_proba(X_test)[:, 1]

final_threshold = float(final_choice["best_threshold_global"])
final_pred = (final_test_proba >= final_threshold).astype(int)

submission = sample_sub.copy()
submission.iloc[:, -1] = final_pred
submission.to_csv("submit.csv", index=False)


In [ ]:
# Final output cell (strict summary)
best_cv_mean = final_choice["f1@best_global"]
best_cv_std = np.std(final_choice.get("fold_best_f1", [best_cv_mean])) if "fold_best_f1" in final_choice else np.nan

print("best approach:", "ensemble" if use_ensemble else "single model")
print(f"best CV F1 mean/std: {best_cv_mean:.6f} / {best_cv_std:.6f}")
print(f"best PR-AUC: {final_choice['pr_auc']:.6f}")
print(f"chosen threshold (global): {final_choice['best_threshold_global']:.2f}")
print("submit.csv saved successfully")
print(submission.head())
print("submit.csv shape:", submission.shape)
